# Real-Time Data Processing with Apache Spark

This tutorial demonstrates how to process real-time data using Apache Spark's Structured Streaming capabilities. It includes reading data from Kafka, enriching it with static data from PostgreSQL, and applying various output modes, windowing, and watermarking strategies to illustrate the flexibility and power of Spark Streaming.

We will:

1. Set up the environment to connect to Kafka and PostgreSQL.
2. Inspect incoming Kafka data.
3. Demonstrate different streaming modes (Append, Complete, and Update).
4. Apply Windowing and Watermarking for time-based aggregations.


In [11]:
import os
from pyspark.sql import SparkSession

# Path to the PostgreSQL JDBC jar file
rel_path = os.path.relpath('./libs/postgresql-42.7.3.jar')

# Connection URL and properties for PostgreSQL
url = "jdbc:postgresql://fhtw-big-data.postgres.database.azure.com/music_store"
postgres_options = {
    "url": url,
    "user": "student",
    "password": "reRZ2pjg1WxqlwjU",
    "driver": "org.postgresql.Driver"
}

# Kafka configuration parameters
kafka_params = {
    "kafka.bootstrap.servers": "pkc-12576z.us-west2.gcp.confluent.cloud:9092",
    "subscribe": "music_streams",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": "org.apache.kafka.common.security.plain.PlainLoginModule required username='TGYVM6H5NCVC5EII' password='ySPBA/RJVdw+k0PgKsgN04Z0VqcqDGKi6RfDqx7Rok/4703E7AX/GjBU12zXs7mP';"
}

# Create a Spark session
spark = SparkSession.builder \
    .appName("PostgreSQL Integration Example") \
    .getOrCreate()

# Inspecting Kafka Stream

To analyze the data, we first read the real-time stream from the Kafka topic `music_streams`. We parse the incoming JSON data into a structured format for further processing.


In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

# Define the schema for the JSON data from Kafka
json_schema = StructType([
    StructField("ts", LongType(), True),
    StructField("auth", StringType(), True),
    StructField("page", StringType(), True),
    StructField("song", StringType(), True),
    StructField("level", StringType(), True),
    StructField("artist", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("method", StringType(), True),
    StructField("status", IntegerType(), True),
    StructField("userId", StringType(), True),
    StructField("lastName", StringType(), True),
    StructField("location", StringType(), True),
    StructField("track_id", IntegerType(), True),
    StructField("firstName", StringType(), True),
    StructField("sessionId", IntegerType(), True),
    StructField("userAgent", StringType(), True),
    StructField("registration", LongType(), True),
    StructField("itemInSession", IntegerType(), True)
])

# Read streaming data from Kafka
df = spark \
    .readStream \
    .format("kafka") \
    .options(**kafka_params) \
    .load()

# Parse the JSON data
df_message = df.selectExpr("CAST(value AS STRING) as json_string")
parsed_df = df_message.withColumn("jsonData", from_json(col("json_string"), json_schema)).select("jsonData.*")

# Display the parsed data
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

import time
time.sleep(30)
query.stop()

# Output Modes: Append, Complete, and Update

Spark's streaming query supports three output modes:

1. **Append**: Outputs only new rows added to the result.
2. **Complete**: Outputs the entire result table on every trigger.
3. **Update**: Outputs only the rows updated in the result.

### Example: Append Mode

Append mode is useful for incremental data processing, where only new events are appended.


In [ ]:
query = parsed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

time.sleep(30)
query.stop()

### Example: Complete Mode

Complete mode is ideal for use cases where the entire result needs to be recalculated.


In [ ]:
levels_counts = parsed_df.groupBy("level").count()

query = levels_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .start()

time.sleep(40)
query.stop()

In [ ]:
# Verarbeite alle 3 Sekunden
query = levels_counts.writeStream \
    .outputMode("complete") \
    .format("console") \
    .trigger(processingTime="3 seconds") \
    .start()

### Example: Update Mode

Update mode is suitable for scenarios where only updated rows are required, such as maintaining metrics.


In [ ]:
query = levels_counts.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

time.sleep(30)
query.stop()

# Windowing and Watermarking

Windowing allows grouping events into time-based intervals. Watermarking manages late-arriving data by defining a threshold for processing.

### Tumbling Window

Fixed, non-overlapping intervals.


In [ ]:
from pyspark.sql.functions import window, from_unixtime

# Convert 'ts' from LongType to TimestampType
parsed_df_with_timestamp = parsed_df.withColumn("ts", from_unixtime("ts").cast("timestamp"))

# Apply windowing
windowed_counts = parsed_df_with_timestamp \
    .groupBy(window("ts", "1 minute"), "level") \
    .count()

# Start the streaming query
query = windowed_counts.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

import time
time.sleep(60)
query.stop()

### Sliding Window

Overlapping intervals for finer granularity.


In [ ]:
windowed_counts = parsed_df_with_timestamp.groupBy(window("ts", "1 minute", "30 seconds"), "level").count()

query = windowed_counts.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

time.sleep(60)
query.stop()

### Watermarking: Append Mode

Watermarking with Append Mode ensures efficient handling of late-arriving events.


In [ ]:
watermarked_df = parsed_df_with_timestamp.withWatermark("ts", "2 minutes").groupBy(window("ts", "1 minute"), "level").count()

query = watermarked_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

time.sleep(60)
query.stop()

### Watermarking: Update Mode

Update Mode with watermarking outputs only updated results while handling late events.


In [ ]:
query = parsed_df_with_timestamp.writeStream \
    .outputMode("update") \
    .format("console") \
    .start()

time.sleep(60)
query.stop()